# V1

## 清 GPU

In [ ]:
# import os
# os._exit(0)

## pull 模型 - kagglehub

In [3]:
import os
import time
from huggingface_hub import snapshot_download

repo_id = "Efficient-Large-Model/Sana_Sprint_1.6B_1024px_diffusers"
local_dir = "/kaggle/working/Sana_Sprint_1.6B_1024px_diffusers"

start = time.perf_counter()

path = snapshot_download(
    repo_id=repo_id,
    local_dir=local_dir,
    max_workers=16,
)

elapsed = time.perf_counter() - start

print(f"耗时：{elapsed:.2f} 秒")
print(f"路径：{path}")
print(f"路径存在：{os.path.exists(path)}")
print(f"model_index.json 存在：{os.path.exists(os.path.join(path, 'model_index.json'))}")

Fetching 17 files:   0%|          | 0/17 [00:00<?, ?it/s]

耗时：26.97 秒
路径：/kaggle/working/Sana_Sprint_1.6B_1024px_diffusers
路径存在：True
model_index.json 存在：True


In [ ]:
# import time
# import os
# import kagglehub

# handle = "kawchar85/sana_sprint_1.6b_1024px/transformers/default/1"

# start = time.perf_counter()

# path = kagglehub.model_download(handle)

# elapsed = time.perf_counter() - start

# print(f"耗时：{elapsed:.2f} 秒")
# print(f"路径：{path}")
# print(f"路径存在：{os.path.exists(path)}")


## Cell 1：加载双卡模型 已 pull

In [4]:
import torch
from diffusers import SanaSprintPipeline

assert torch.cuda.device_count()>=2,f"需要2张GPU，当前只有{torch.cuda.device_count()}张"

def load_pipe(i):
    d=f"cuda:{i}"; p=SanaSprintPipeline.from_pretrained(path,torch_dtype=torch.bfloat16,local_files_only=True); p.to(d); p.vae.to(device=d,dtype=torch.float32); print(f"GPU{i} {torch.cuda.get_device_name(i)} | {torch.cuda.memory_allocated(i)/1024**3:.2f}GiB"); return p

pipe0,pipe1=load_pipe(0),load_pipe(1)

Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

GPU0 Tesla T4 | 9.12GiB


Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

GPU1 Tesla T4 | 9.12GiB


## Cell 2：只改 Prompt 和参数

In [ ]:
OUT="/kaggle/working/sana_output"; W=H=1024; STEPS=2; BASE_SEED=20260819

PROMPTS=[
    "Majestic fjord cutting through steep granite cliffs, deep emerald water reflecting towering peaks, mist clinging to rock faces, Nordic minimalism meets dramatic scale, crisp cold atmosphere, no people",
    
    "Sunflower field at golden hour, endless sea of yellow heads turning toward setting sun, warm amber light creating long shadows, impressionist painting vibe with soft focus edges, vibrant and joyful energy, no people",
    
    "Ancient redwood forest cathedral, massive tree trunks rising into canopy, shafts of sunlight piercing through dense fog, moss-covered ground, sense of awe and primordial silence, vertical composition emphasizing height, no people",
    
    "Salt flats after rain creating perfect mirror surface, infinite sky reflected on ground, solitary tree in distance breaking symmetry, surreal minimalist landscape, dreamlike disorientation between earth and sky, no people",
    
    "Glacier calving into icy blue lagoon, chunks of ice floating like sculptures, turquoise water contrasting with white snow, raw power of nature, climate change narrative, sharp details of ice textures, no people",
    
    "Lavender fields in Provence rolling hills, purple waves stretching to horizon, rustic stone farmhouse in background, soft hazy summer light, romantic pastoral scene, Monet-inspired color palette, no people",
    
    "Dramatic lightning storm over prairie landscape, electric bolts illuminating dark clouds, green grass bending in wind, tension between calm earth and chaotic sky, high contrast black and white aesthetic, no people",
    
    "Coral reef underwater panorama, vibrant tropical fish swimming among colorful corals, sunlight filtering through clear blue water from above, kaleidoscope of marine life, National Geographic style, no people",
    
    "Autumn vineyard in Tuscany, rows of grapevines with golden and crimson leaves, rolling hills dotted with cypress trees, warm harvest light, rustic elegance and agricultural beauty, no people",
    
    "Meteor shower over silent desert landscape, multiple shooting stars streaking across Milky Way, ancient rock formations silhouetted against starry sky, cosmic connection and earthly permanence, astrophotography masterpiece, no people",
]

## Cell 3：双卡 Dispatcher 执行 + 异步传输队列

In [ ]:
# !pip install -q aiohttp cryptography

import os,time,queue,threading,asyncio,io,hashlib,torch,aiohttp
from cryptography.hazmat.primitives.ciphers.aead import AESGCM

UPLOAD_URL="https://ranran-sana.202820.xyz/upload"; PASSWORD="wangran"; KEY=hashlib.sha256(PASSWORD.encode()).digest()
W=H=1024; STEPS=2; BASE_SEED=20260819; IO_WORKERS=4
tasks,uploads=queue.Queue(),queue.Queue(maxsize=32)

# 图片编码为 WebP
def encode(image):
    b=io.BytesIO(); image.save(b,"WEBP",quality=90,method=4); return b.getvalue()

# AES-GCM 加密图片
def encrypt(data):
    nonce=os.urandom(12); return nonce+AESGCM(KEY).encrypt(nonce,data,None)

# 编码、加密并异步上传一张图片
async def upload(x,session):
    try:
        data=await asyncio.to_thread(encode,x["image"]); data=encrypt(data)
        form=aiohttp.FormData(); form.add_field("file",data,filename=f'{x["id"]:04d}.bin',content_type="application/octet-stream")
        for k in ("id","gpu","seed","prompt","seconds"): form.add_field(k,str(x[k]))
        async with session.post(UPLOAD_URL,data=form) as r:
            if r.status>=400: raise RuntimeError(f"HTTP {r.status}: {await r.text()}")
        print(f'↑ #{x["id"]:03d} | GPU{x["gpu"]} → PC')
    except Exception as e: print(f'! #{x["id"]:03d} 上传失败: {e}')
    finally: x.pop("image",None); uploads.task_done()

# 后台异步上传 Worker
async def io_worker(session):
    while True:
        x=await asyncio.to_thread(uploads.get)
        if x is None: uploads.task_done(); break
        await upload(x,session)

# 创建 aiohttp 会话和多个上传 Worker
async def io_main():
    async with aiohttp.ClientSession(headers={"Authorization":f"Bearer {PASSWORD}"},timeout=aiohttp.ClientTimeout(total=60)) as session:
        await asyncio.gather(*(io_worker(session) for _ in range(IO_WORKERS)))

# 单独线程运行异步网络层
def run_io(): asyncio.run(io_main())

# GPU Worker 只负责生成，生成完立即交给上传队列
def gpu_worker(i,p):
    d=f"cuda:{i}"; torch.cuda.set_device(i)
    while True:
        task=tasks.get()
        if task is None: tasks.task_done(); break
        idx,prompt,seed=task; start=time.perf_counter()
        try:
            g=torch.Generator(device=d).manual_seed(seed)
            with torch.inference_mode(): image=p(prompt=prompt,num_inference_steps=STEPS,guidance_scale=0.0,width=W,height=H,generator=g).images[0]
            sec=time.perf_counter()-start; uploads.put({"id":idx,"gpu":i,"seed":seed,"prompt":prompt,"seconds":round(sec,3),"image":image})
            print(f"✓ #{idx:03d} | GPU{i} | {sec:.2f}s")
        except Exception as e: print(f"✗ #{idx:03d} | GPU{i} | {e}")
        finally: tasks.task_done()

io_thread=threading.Thread(target=run_io,daemon=True); io_thread.start()
workers=[threading.Thread(target=gpu_worker,args=(0,pipe0),daemon=True),threading.Thread(target=gpu_worker,args=(1,pipe1),daemon=True)]
for w in workers: w.start()

start=time.perf_counter()
for i,p in enumerate(PROMPTS): tasks.put((i,p,BASE_SEED+i))
tasks.join()
for _ in workers: tasks.put(None)
tasks.join()
for w in workers: w.join()

uploads.join()
for _ in range(IO_WORKERS): uploads.put(None)
uploads.join(); io_thread.join()

total=time.perf_counter()-start
print(f"\nDONE | {len(PROMPTS)} images | {total:.2f}s | {len(PROMPTS)/total:.3f} img/s")

# v2

## Cell 2：双卡 Dispatcher

In [5]:
# !pip install -q aiohttp cryptography

import os,time,queue,threading,asyncio,io,hashlib,torch,aiohttp
from cryptography.hazmat.primitives.ciphers.aead import AESGCM

BASE="https://ranran-sana.202820.xyz"; TASK_URL=f"{BASE}/task/next"; UPLOAD_URL=f"{BASE}/upload"
PASSWORD="wangran"; KEY=hashlib.sha256(PASSWORD.encode()).digest(); IO_WORKERS=4
tasks=queue.Queue(maxsize=32); uploads=queue.Queue(maxsize=64); STOP=threading.Event()

# PIL 图片编码 WebP
def encode(image):
    b=io.BytesIO(); image.save(b,"WEBP",quality=90,method=4); return b.getvalue()

# AES-GCM 加密图片
def encrypt(data):
    nonce=os.urandom(12); return nonce+AESGCM(KEY).encrypt(nonce,data,None)

# 异步编码、加密并上传图片
async def upload_image(x,session):
    try:
        data=await asyncio.to_thread(encode,x["image"]); data=await asyncio.to_thread(encrypt,data)
        form=aiohttp.FormData(); form.add_field("file",data,filename=f'{x["id"]:04d}.bin',content_type="application/octet-stream")
        for k in ("id","gpu","seed","prompt","seconds"): form.add_field(k,str(x[k]))
        async with session.post(UPLOAD_URL,data=form) as r:
            if r.status>=400: raise RuntimeError(f"HTTP {r.status}: {await r.text()}")
        print(f'↑ #{x["id"]:03d} | GPU{x["gpu"]} → PC')
    except Exception as e: print(f'! Upload #{x["id"]:03d}: {e}')
    finally: x.pop("image",None); uploads.task_done()

# 异步上传消费者
async def upload_worker(session):
    while True:
        x=await asyncio.to_thread(uploads.get)
        if x is None: uploads.task_done(); break
        await upload_image(x,session)

# 长轮询本地 Prompt
async def task_poller(session):
    while not STOP.is_set():
        try:
            async with session.get(TASK_URL) as r:
                if r.status==204: continue
                if r.status>=400: raise RuntimeError(f"HTTP {r.status}: {await r.text()}")
                x=await r.json(); await asyncio.to_thread(tasks.put,x); print(f'↓ #{x["id"]:03d} | {x["prompt"][:70]}')
        except Exception as e:
            if not STOP.is_set(): print(f"! Poller: {e}"); await asyncio.sleep(2)

# 网络层：Task Pull + Result Upload
async def network_main():
    timeout=aiohttp.ClientTimeout(total=35)
    async with aiohttp.ClientSession(headers={"Authorization":f"Bearer {PASSWORD}"},timeout=timeout) as session:
        await asyncio.gather(task_poller(session),*(upload_worker(session) for _ in range(IO_WORKERS)))

# 独立线程运行异步网络层
def run_network(): asyncio.run(network_main())

# GPU Worker：取任务、生成、立即交给上传队列
def gpu_worker(i,p):
    d=f"cuda:{i}"; torch.cuda.set_device(i)
    while True:
        x=tasks.get()
        if x is None: tasks.task_done(); break
        start=time.perf_counter()
        try:
            g=torch.Generator(device=d).manual_seed(x["seed"])
            with torch.inference_mode(): image=p(prompt=x["prompt"],num_inference_steps=x["steps"],guidance_scale=0.0,width=x["width"],height=x["height"],generator=g).images[0]
            sec=round(time.perf_counter()-start,3); item={"id":x["id"],"gpu":i,"seed":x["seed"],"prompt":x["prompt"],"seconds":sec,"image":image}
            try: uploads.put_nowait(item)
            except queue.Full: print(f'! Upload Queue 满，丢弃 #{x["id"]}')
            print(f'✓ #{x["id"]:03d} | GPU{i} | {sec:.2f}s')
        except Exception as e: print(f'✗ #{x["id"]:03d} | GPU{i} | {e}')
        finally: tasks.task_done()

# 启动远程消费者和双 GPU Dispatcher
network_thread=threading.Thread(target=run_network,daemon=True); network_thread.start()
gpu_workers=[threading.Thread(target=gpu_worker,args=(0,pipe0),daemon=True),threading.Thread(target=gpu_worker,args=(1,pipe1),daemon=True)]
for w in gpu_workers: w.start()
print("✓ SANA Remote Dispatcher 已启动 | GPU0 + GPU1 | 等待本地 Prompt...")

✓ SANA Remote Dispatcher 已启动 | GPU0 + GPU1 | 等待本地 Prompt...
↓ #001 | A colossal, moss-covered ancient mechanical robot sleeping deep in a l
↓ #002 | Whimsical steampunk floating islands in the sky, waterfalls cascading 
↓ #003 | A majestic, cosmic star-whale swimming gracefully through a sea of glo
↓ #004 | An antique steam train traveling across an infinite frozen ocean at ni
↓ #005 | A colossal, ancient open-air library carved into a massive red rock ca
↓ #006 | A surreal landscape where the sky and water perfectly merge, giant inv
↓ #007 | A retro-futuristic abandoned amusement park built inside a glowing met


  0%|          | 0/2 [00:00<?, ?it/s]

  0%|          | 0/2 [00:00<?, ?it/s]

↓ #008 | An alien desert landscape under twin moons, a gigantic glowing crystal
↓ #009 | A mystical underground cavern filled with giant glowing bioluminescent
↓ #010 | A surreal corridor made of floating, glowing picture frames in the sky
✓ #002 | GPU1 | 7.75s
✓ #001 | GPU0 | 8.29s


  0%|          | 0/2 [00:00<?, ?it/s]

...

  0%|          | 0/2 [00:00<?, ?it/s]

↑ #004 | GPU0 → PC
✓ #007 | GPU1 | 5.04s
✓ #008 | GPU0 | 5.01s


  0%|          | 0/2 [00:00<?, ?it/s]

  0%|          | 0/2 [00:00<?, ?it/s]

↑ #003 | GPU1 → PC
✓ #009 | GPU1 | 5.07s
✓ #010 | GPU0 | 5.04s
↑ #006 | GPU0 → PC
↑ #005 | GPU1 → PC
↑ #007 | GPU1 → PC
↑ #008 | GPU0 → PC
↑ #009 | GPU1 → PC
↑ #010 | GPU0 → PC
